# Chapter 5 — Retrieval at Scale with Approximate Nearest Neighbors

*Companion notebook for* **Modern Recommender Systems** *(Manning), Chapter 5.*

This notebook demonstrates the ANN retrieval components of Section 5.2. The
implementations live in
`recsys.fourstage_recsys.retrieval.ann_retrieval`:

- `build_index` / `query_index` — the minimal workflow (Listing 5.4)
- `ANNRetrievalIndex` — index configuration, incremental updates, recall
  measurement against exact search, switching among flat, HNSW, and IVF
- `warm_start_embedding` — the retrieval cold-start strategy (Section 5.2.4)

**Run `ch05_similarity_learning.ipynb` first** — this notebook loads the
embeddings it saves to `data/processed/chapter05/`.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "recsys").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import json
import time

import numpy as np
import pandas as pd

from recsys.fourstage_recsys.retrieval.ann_retrieval import (
    ANNRetrievalIndex, build_index, query_index, warm_start_embedding,
)

np.random.seed(42)

ART = project_root / "data" / "processed" / "chapter05"
assert (ART / "embeddings.npz").exists(), (
    "Artifacts not found — run ch05_similarity_learning.ipynb first."
)
data = np.load(ART / "embeddings.npz")
query_embeddings = data["infonce_query"]
item_embeddings = data["infonce_cand"]
with open(ART / "mappings.json") as fh:
    mappings = json.load(fh)
item_ids = mappings["item_ids"]
idx_to_title = {int(k): v for k, v in mappings["idx_to_title"].items()}
idx_to_genres = {int(k): v for k, v in mappings["idx_to_genres"].items()}
anchor_idx = mappings["anchor_idx"]
num_items, emb_dim = item_embeddings.shape
print(f"{num_items:,} items, {emb_dim}-dimensional embeddings")

## 1. Building and querying an HNSW index (Listing 5.4)

The workflow is three steps: build an index from item embeddings, query it
with a query embedding, and map the results back to item IDs. Two things the
package implementation is pedantic about:

1. **Normalize consistently** between training, index construction, and
   query time. When embeddings are L2-normalized, the inner product equals
   cosine similarity — and failing to normalize at any one of the three
   stages is among the most common sources of silent quality degradation in
   production retrieval.
2. **Set the metric explicitly.** `faiss.IndexHNSWFlat(dim, 32)` defaults to
   `METRIC_L2`, not inner product. With normalized vectors the *ranking* is
   identical (squared L2 distance is 2 − 2·cos), so the bug is invisible in
   recall metrics — but the returned scores are distances (smaller is
   better), which silently breaks anything downstream that treats them as
   similarities.

In [ ]:
hnsw_index = build_index(item_embeddings)
print(f"HNSW index built with {hnsw_index.ntotal:,} vectors")

## 2. Spot check: query the index with the anchor movie (Listing 5.5)

Toy Story's nearest neighbors should be animated and family films. If they
are not, something upstream is broken — usually the normalization or an
ID-mapping bug.

In [ ]:
ids, scores = query_index(hnsw_index, query_embeddings[anchor_idx], k=6)
print(f"{idx_to_title[anchor_idx]} neighbors:")
for i, s in zip(ids, scores):
    if i == anchor_idx:
        continue                               # the seed retrieves itself
    print(f"  {s:.3f}  {idx_to_title[int(i)]}")

## 3. Recall and latency across index types

`ANNRetrievalIndex` wraps FAISS with the pieces production needs: a
bidirectional ID mapping, incremental additions, post-retrieval filtering in
place of deletion (FAISS cannot remove vectors from most index types), and
`recall_vs_exact` — the acceptance test to run before deploying any index
configuration (0.95+ against exact search is typically acceptable).

The trade-off from Section 5.2.1, measured: flat is exact but O(N·d) per
query; HNSW buys large speedups at high recall but is slower to build; IVF
builds fast and updates well at somewhat lower recall. On a catalog this
small the absolute latencies are all sub-millisecond — the *relative*
picture is what scales.

In [ ]:
test_queries = query_embeddings[
    np.random.choice(num_items, size=200, replace=False)]

rows = []
for index_type in ["flat", "hnsw", "ivf"]:
    idx = ANNRetrievalIndex(index_type=index_type, emb_dim=emb_dim,
                            ivf_nlist=min(100, max(4, num_items // 40)))
    t0 = time.perf_counter()
    idx.add_items(item_ids, item_embeddings)
    build_ms = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter()
    for q in test_queries:
        idx.retrieve(q, k=100)
    query_ms = (time.perf_counter() - t0) * 1000 / len(test_queries)
    recall = idx.recall_vs_exact(test_queries, k=100)
    rows.append({"index": index_type, "build (ms)": round(build_ms, 1),
                 "query (ms)": round(query_ms, 3),
                 "recall@100 vs exact": round(recall, 4)})

pd.DataFrame(rows).set_index("index")

One caveat when reading the HNSW recall number on a small catalog: clustered
embeddings produce many near-ties, which makes the exact top-100 partly
arbitrary — the ANN index gets "penalized" for returning items with
identical scores in a different order. On the full MovieLens sample, HNSW at
these settings sits comfortably above 0.95; if it does not, raise
`efSearch` and remeasure.

## 4. Cold start: warm-starting a new item (Section 5.2.4)

A brand-new item has no interactions, so the trained model cannot give it an
embedding — and an item without an embedding is invisible to ANN retrieval.
`warm_start_embedding` initializes the new item as the average of existing
items that share its genres, and the incremental `add_items` makes it
retrievable immediately.

In [ ]:
genre_to_items = {}
for i, genres in idx_to_genres.items():
    for genre in genres:
        genre_to_items.setdefault(genre, []).append(i)

ann = ANNRetrievalIndex(index_type="hnsw", emb_dim=emb_dim)
ann.add_items(item_ids, item_embeddings)

sample_genre = idx_to_genres[anchor_idx][0]
new_embedding = warm_start_embedding([sample_genre], genre_to_items,
                                     item_embeddings)
ann.add_items(["new_item_0"], new_embedding.reshape(1, -1))

title_of = {mid: idx_to_title[i] for i, mid in enumerate(item_ids)}
print(f"A new '{sample_genre}' item lands next to:")
for item_id, score in ann.retrieve(new_embedding, k=5,
                                   exclude={"new_item_0"}):
    print(f"  {score:.3f}  {title_of[item_id]}")

The new item is retrievable the moment it is added, and it lands in roughly
the right neighborhood. The embedding cannot capture what makes this
particular item distinctive — the next training cycle, once real
interactions accumulate, will replace it with a behavioral embedding.

## Summary

ANN search trades a small, *measurable* amount of recall for query times
that are orders of magnitude faster than exact search — and "measurable" is
the operative word: `recall_vs_exact` is the acceptance test for any index
configuration change. In `ch05_full_pipeline.ipynb`, this index becomes the
retrieval stage of the four-stage recommender.